# 17 ROUTE: operating distance, motion at onset and where faults are logged

The data carry distance and speed per second (June 2025 only) and a named location for every logged message, but no gradient and no train load (passenger rakes of about 22 coaches, about 1,400 t; route records are kept for 15 days only).

1. June distance run, energised hours and onsets per 1,000 km, by fault group.
2. Speed at onset: moving or standing, by fault group and category.
3. Where onsets are logged over the full log: leading locations and coordinates, with the share of the blower message, and the home shed.

Outputs: `OUTPUT/v4/rev_route_*.csv`. The category grouping in step 2 is an earlier, six-category version; the paper uses the seven categories of Table 1. Table 3 is built from the coordinates written here by `TABLE3_STATES.py`.

Paper: Sections 3.1 and 4.5; Table 3.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

V = "v4"            # data version: corrected log with the explicit exclusion rule (REV_REBUILD_v4)
env = os.environ.get
A1, A2 = "../ANALYSIS1", "../ANALYSIS2"
BUILD, DATA = env("REV_BUILDDIR", "build"), env("REV_DATADIR", "DATA")
CFG = dict(
    fault_log = f"{DATA}/37011FL{V}.csv",
    episodes  = f"{BUILD}/fault_episodes_{V}.csv",
    labelled  = f"{BUILD}/telemetry_labelled_{V}.pkl",
    july      = f"{BUILD}/telemetry_labelled_july_{V}.pkl",
    geo       = f"{A1}/DATA/37011/37011GEO.csv",
    outdir    = env("REV_OUTDIR", "OUTPUT") + f"/{V}",
    blower    = "AUX CAP MACH RM BLOW1 NOT OFF",
    june      = ("2025-06-01 00:00:00", "2025-06-30 23:59:59"),
    windows   = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    seeds     = [int(s) for s in env("REV_SEED_LIST", "42,1,2,3,4").split(",")],
    patience  = 100, test_frac = 0.20, val_frac = 0.20,
)
CATEGORY = {0: "Ventilation", 2: "Protection system", 3: "Protection system",
            5: "Traction and power conversion", 6: "Traction and power conversion",
            7: "Traction and power conversion", 9: "Braking and pneumatic",
            10: "Protection system", 11: "Protection system",
            12: "Traction and power conversion", 13: "Protection system",
            14: "Traction and power conversion", 15: "Auxiliary",
            17: "Power supply quality", 19: "Power supply quality",
            16: "Auxiliary", 8: "Protection system"}   # restored in v4: ASCU messages; Fire Alarm
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE",
           "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4",
              "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| data", V)

## 1. Distance and exposure in June

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
j0, j1 = (pd.Timestamp(x) for x in CFG["june"])
jun = ep[ep.onset.between(j0, j1)].copy()
jun["group"] = np.where(jun.fault.eq(CFG["blower"]), "blower", "other labels")

km = tel["DIST_M"].sum() / 1000
hours = len(tel) / 3600
moving_h = (tel["KMH"] > 0).sum() / 3600
print(f"June: {km:,.0f} km run in {hours:,.1f} energised hours ({moving_h:,.1f} h moving)")
rows = [dict(group="all labels", onsets=len(jun))] + \
       [dict(group=g, onsets=int((jun.group == g).sum())) for g in ["blower", "other labels"]]
expo = pd.DataFrame(rows)
expo["per_1000_km"] = expo.onsets / km * 1000
expo["per_energised_hour"] = expo.onsets / hours
print(expo.round(2).to_string(index=False))
expo.to_csv(f"{CFG['outdir']}/rev_route_exposure.csv", index=False)

## 2. Speed at onset

Speed is read at the onset second (or, failing that, the second before). An onset counts
as standing if the speed is zero.

In [ ]:
pos = pd.Series(np.arange(len(tel)), index=pd.DatetimeIndex(tel["DATETIME"].values))
i0 = pos.reindex(pd.DatetimeIndex(jun.onset))
i1 = pos.reindex(pd.DatetimeIndex(jun.onset) - pd.Timedelta(seconds=1))
idx = np.where(i0.notna().values, i0.values, i1.values)
jun["kmh"] = [tel["KMH"].iat[int(i)] if not np.isnan(i) else np.nan for i in idx]
jun["category"] = jun.cluster.map(CATEGORY).fillna("unassigned")
jun["state"] = np.where(jun.kmh.isna(), "no telemetry", np.where(jun.kmh > 0, "moving", "standing"))
by_g = pd.crosstab(jun.group, jun.state, margins=True)
by_c = pd.crosstab(jun.category, jun.state, margins=True)
for t in (by_g, by_c):
    t["moving_pct"] = (100 * t.get("moving", 0) / (t.get("moving", 0) + t.get("standing", 0))).round(1)
print(by_g.to_string(), "\n"); print(by_c.to_string())
pd.concat({"group": by_g, "category": by_c}).to_csv(f"{CFG['outdir']}/rev_route_motion.csv")

## 3. Where onsets are logged (full log)

Each onset record carries the location written in the log. Locations are also grouped by
the coordinates of `37011GEO.csv`, since several names can share one coordinate pair.

In [ ]:
fl = pd.read_csv(CFG["fault_log"])
on = fl[~fl.FAULT.str.contains("Recovered", case=False)].copy()
on["blower"] = on.FAULT.str.strip().str.upper().eq(CFG["blower"])
print(f"onset records: {len(on):,} (episodes: {len(ep):,})")

top = (on.groupby("LOCATION").agg(onsets=("FAULT", "size"), blower=("blower", "sum"))
         .sort_values("onsets", ascending=False))
top["share_pct"] = (100 * top.onsets / len(on)).round(1)
top["blower_pct"] = (100 * top.blower / top.onsets).round(1)
print(top.head(12).to_string(), "\n")
top.to_csv(f"{CFG['outdir']}/rev_route_locations.csv")

for name, pat in [("Santragachi (home shed)", "SANTRAGACHI"), ("Howrah (stabling)", "HWH|HOWRAH")]:
    m = on.LOCATION.str.contains(pat, case=False, na=False)
    print(f"{name:24s} {int(m.sum()):5d} onsets ({100*m.mean():.1f}%), blower {100*on[m].blower.mean():.1f}%, "
          f"names: {sorted(on[m].LOCATION.unique())}")
print(f"{'elsewhere':24s} blower {100*on[~on.LOCATION.str.contains('SANTRAGACHI|HWH|HOWRAH', case=False, na=False)].blower.mean():.1f}%")

geo = pd.read_csv(CFG["geo"])
geo = geo.rename(columns={c: c.upper() for c in geo.columns})
lat = next(c for c in geo.columns if c.startswith("LAT")); lon = next(c for c in geo.columns if c.startswith("LON"))
on = on.merge(geo[["LOCATION", lat, lon]].drop_duplicates("LOCATION"), on="LOCATION", how="left")
coord = (on.dropna(subset=[lat]).groupby([lat, lon])
           .agg(onsets=("FAULT", "size"), blower=("blower", "sum"),
                names=("LOCATION", lambda s: "; ".join(sorted(s.unique()))))
           .sort_values("onsets", ascending=False))
coord["blower_pct"] = (100 * coord.blower / coord.onsets).round(1)
print(f"\nonset records with coordinates: {int(on[lat].notna().sum()):,}")
print(coord.head(6).to_string())
coord.to_csv(f"{CFG['outdir']}/rev_route_coordinates.csv")

___
**END**